# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Role of Explainability
This is the final analytical stage of the pipeline (Stage 11). While previous stages proved *how well* the model performs and *where* it makes mistakes, this stage proves *why* the model makes its decisions. Explainable AI (XAI) bridges the gap between mathematical optimization and human linguistic interpretation.


# 2. EXPLAINABILITY OBJECTIVES

Our goals for this stage are to:
- Identify the most important Indonesian words/n-grams that the model associates with cyberbullying (Global Explainability).
- Extract the top vocabulary features specific to each cyberbullying class.
- Explain individual predictions (Local Explainability) using LIME to understand why a specific sentence was flagged.
- Connect explainability back to our Error Analysis (Stage 10) to understand why the model was "confused" on certain texts.
- Provide a clear interpretation framework while acknowledging the technical limitations of TF-IDF.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import json
import joblib
import warnings

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn for matrix handling
from sklearn.pipeline import make_pipeline

# XAI Libraries
try:
    from lime.lime_text import LimeTextExplainer
except ImportError:
    print("Warning: 'lime' is not installed. Run 'pip install lime' for local explainability.")

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
MODELS_DIR = PROJECT_ROOT / "models"
BASE_REPORTS_DIR = PROJECT_ROOT / "reports"

# Explainability specific reports directory
EXPLAIN_DIR = REPORTS_DIR / "explainability"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)

# Data Paths
X_TEST_PATH = TFIDF_DIR / "X_test_tfidf.npz"
Y_TEST_PATH = TFIDF_DIR / "y_test.csv"
TEXT_METADATA_PATH = TFIDF_DIR / "test_metadata.csv"
VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"
MODEL_SELECTION_PATH = BASE_REPORTS_DIR / "model_evaluation" / "model_selection.json"
ERROR_ANALYSIS_PATH = REPORTS_DIR / "error_analysis" / "misclassified_samples.csv"

# Configuration parameters
TOP_N_FEATURES = 20

print(f"Explainability Reports Directory: {EXPLAIN_DIR}")


Explainability Reports Directory: /home/zapp/Kampus/PM-NEW/reports/explainability


In [3]:
# 5. LOAD TF-IDF ARTIFACTS

if not VECTORIZER_PATH.exists():
    raise FileNotFoundError("FAIL: TF-IDF Vectorizer not found. Please run 06_tfidf.ipynb first.")

print("Loading Fitted TF-IDF Vectorizer...")
tfidf_vectorizer = joblib.load(VECTORIZER_PATH)
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

print(f"Vocabulary Size: {len(feature_names)} features (words/n-grams)")


Loading Fitted TF-IDF Vectorizer...
Vocabulary Size: 35222 features (words/n-grams)


In [4]:
# 6. LOAD SELECTED MODEL

if not MODEL_SELECTION_PATH.exists():
    raise FileNotFoundError("FAIL: model_selection.json not found. You must complete 09_model_evaluation.ipynb.")

with open(MODEL_SELECTION_PATH, 'r') as f:
    selection_meta = json.load(f)

best_model_name = selection_meta['selected_model']
print(f"Targeting Best Model: {best_model_name}")

model_path = MODELS_DIR / f"{best_model_name}.pkl"
best_model = joblib.load(model_path)

# Handle XGBoost Mapping if necessary
xgb_mapping = None
if 'xgboost' in best_model_name:
    xgb_mapping_path = MODELS_DIR / "xgboost_label_mapping.json"
    with open(xgb_mapping_path, 'r') as f:
        xgb_mapping = json.load(f)
    xgb_reverse_mapping = {v: k for k, v in xgb_mapping.items()}
    class_labels = list(xgb_mapping.keys())
else:
    class_labels = list(best_model.classes_)
    
print(f"Model loaded successfully. Recognized {len(class_labels)} classes.")


Targeting Best Model: linear_svm_baseline
Model loaded successfully. Recognized 4 classes.


In [5]:
# 7. LOAD TEST DATA & 8. VALIDATE DATA ALIGNMENT

print("Loading Test Data for contextual explanation...")
y_test_df = pd.read_csv(Y_TEST_PATH)
metadata_df = pd.read_csv(TEXT_METADATA_PATH)

if y_test_df.shape[0] != metadata_df.shape[0]:
    raise ValueError("CRITICAL ERROR: Data Alignment mismatch between labels and text metadata.")

print("Data alignment verified.")


Loading Test Data for contextual explanation...
Data alignment verified.


In [6]:
# 9. IDENTIFY MODEL TYPE

is_linear = hasattr(best_model, "coef_")
is_tree = hasattr(best_model, "feature_importances_")

if is_linear:
    print(f"Model Type: LINEAR (e.g., Logistic Regression, LinearSVC)")
    print("Explainability Method: Model Coefficients (Weight mapping)")
elif is_tree:
    print(f"Model Type: TREE-BASED (e.g., XGBoost, Random Forest)")
    print("Explainability Method: Feature Importances (Gini/Split weight mapping)")
else:
    print("Model Type: UNKNOWN or Black-Box. Global explainability may require SHAP.")


Model Type: LINEAR (e.g., Logistic Regression, LinearSVC)
Explainability Method: Model Coefficients (Weight mapping)


In [7]:
# 10. GLOBAL FEATURE IMPORTANCE & 11. CLASS-SPECIFIC FEATURE IMPORTANCE

def extract_linear_importance(model, feature_names, class_labels):
    importance_records = []
    
    # model.coef_ shape is usually (n_classes, n_features)
    for class_idx, class_name in enumerate(class_labels):
        if len(class_labels) == 2: # Binary classification edge case
            coefs = model.coef_[0]
            if class_idx == 0: coefs = -coefs # Negative class
        else:
            coefs = model.coef_[class_idx]
            
        # Get top N indices sorted by absolute coefficient magnitude (or just positive)
        # We focus on words that strongly *push* towards this class (positive coefs)
        top_positive_indices = coefs.argsort()[-TOP_N_FEATURES:][::-1]
        
        for rank, idx in enumerate(top_positive_indices):
            importance_records.append({
                'Class': class_name,
                'Feature': feature_names[idx],
                'Weight': coefs[idx],
                'Direction': 'Positive',
                'Rank': rank + 1
            })
            
    return pd.DataFrame(importance_records)

def extract_tree_importance(model, feature_names, class_labels):
    # Tree models usually provide a single global array of importances, not per-class
    importances = model.feature_importances_
    top_indices = importances.argsort()[-TOP_N_FEATURES:][::-1]
    
    importance_records = []
    for rank, idx in enumerate(top_indices):
        importance_records.append({
            'Class': 'GLOBAL (All Classes)',
            'Feature': feature_names[idx],
            'Weight': importances[idx],
            'Direction': 'Positive (Gini/Split)',
            'Rank': rank + 1
        })
    return pd.DataFrame(importance_records)

# Extract based on model type
if is_linear:
    df_importance = extract_linear_importance(best_model, feature_names, class_labels)
    print("Extracted Class-Specific Linear Coefficients.")
elif is_tree:
    df_importance = extract_tree_importance(best_model, feature_names, class_labels)
    print("Extracted Global Tree Feature Importances.")

display(df_importance.head(10))

# Save
imp_path = EXPLAIN_DIR / "class_feature_importance.csv"
df_importance.to_csv(imp_path, index=False)


Extracted Class-Specific Linear Coefficients.


,Class,Feature,Weight,Direction,Rank
0,harassment,syiah,3.590835,Positive,1
1,harassment,israel,2.793185,Positive,2
2,harassment,gay,2.755753,Positive,3
3,harassment,kadrun,2.671398,Positive,4
4,harassment,lgbt,2.665307,Positive,5
5,harassment,zionis,2.536161,Positive,6
6,harassment,setara,2.238335,Positive,7
7,harassment,untung aku,2.210035,Positive,8
8,harassment,oleh pasukan,2.168297,Positive,9
9,harassment,rempang,2.104001,Positive,10


In [8]:
# 12. SHAP EXPLAINABILITY

print("SHAP analysis skipped: For sparse TF-IDF text matrices, SHAP requires dense conversion which frequently causes memory exhaustion (OOM) on standard hardware. We rely on Native Model Weights (Global) and LIME (Local) for safe, interpretable NLP analysis.")


SHAP analysis skipped: For sparse TF-IDF text matrices, SHAP requires dense conversion which frequently causes memory exhaustion (OOM) on standard hardware. We rely on Native Model Weights (Global) and LIME (Local) for safe, interpretable NLP analysis.


In [9]:
# 13. LIME EXPLAINABILITY SETUP

print("Setting up LIME (Local Interpretable Model-agnostic Explanations)...")

# To use LIME, we need a prediction function that takes raw text (list of strings)
# and outputs probabilities (shape: n_samples, n_classes).

def lime_predict_pipeline(texts):
    # 1. Transform text using our fitted TF-IDF
    X_vec = tfidf_vectorizer.transform(texts)
    
    # 2. Predict probabilities using the model
    if hasattr(best_model, "predict_proba"):
        probs = best_model.predict_proba(X_vec)
    elif hasattr(best_model, "decision_function"):
        # Convert decision function to mock probabilities using softmax-like scaling
        decisions = best_model.decision_function(X_vec)
        exp_d = np.exp(decisions - np.max(decisions, axis=1, keepdims=True))
        probs = exp_d / np.sum(exp_d, axis=1, keepdims=True)
    else:
        raise ValueError("Model does not support predict_proba or decision_function.")
    
    return probs

# Initialize Explainer
try:
    explainer = LimeTextExplainer(class_names=class_labels)
    lime_ready = True
    print("LIME Explainer initialized successfully.")
except NameError:
    lime_ready = False
    print("LIME could not be initialized because the library is missing.")


Setting up LIME (Local Interpretable Model-agnostic Explanations)...
LIME Explainer initialized successfully.


In [13]:
# 14. LOCAL PREDICTION EXPLANATIONS

if lime_ready:
    # Pick a random sample from the test set for demonstration
    sample_idx = 42
    sample_text = metadata_df.iloc[sample_idx, 0] # Original text
    actual_label = y_test_df.iloc[sample_idx, 0]
    
    print(f"--- LOCAL EXPLANATION EXAMPLE ---")
    print(f"Text: '{sample_text}'")
    print(f"Actual Label: {actual_label}")
    
    # Generate Explanation
    exp = explainer.explain_instance(
        sample_text, 
        lime_predict_pipeline, 
        num_features=6, 
        top_labels=1
    )
    
    predicted_label_idx = exp.available_labels()[0]
    predicted_label = class_labels[predicted_label_idx]
    print(f"Predicted Label: {predicted_label}\n")
    
    print("Words driving this prediction:")
    for feature, weight in exp.as_list(label=predicted_label_idx):
        print(f" - '{feature}' : {weight:.4f}")
        
    # We save the HTML visual locally
    html_path = EXPLAIN_DIR / "lime_sample_explanation.html"
    exp.save_to_file(html_path)
    print(f"\nInteractive LIME HTML visualization saved to: {html_path}")
else:
    print("LIME is not available.")


--- LOCAL EXPLANATION EXAMPLE ---
Text: 'Laporan PERIHAL : TELAH TERJADI PEMBACOKAN TERHADAP WARGA YANG MELINTAS DI JALAN SERTA PENGUNJUNG WARKOP BENK KOPI. Assalamu'alaikum Wr. Wb. dan Kanit Reskrim Polsek Syiah Kuala Aipda Samsuardi dan Anggota piket Polsek Syiah Kuala. Demikian komandan yang dapat kami laporkan, @sumber dari Kapolsek Syiah Kuala @sorotan @pengikut'
Actual Label: normal
Predicted Label: normal

Words driving this prediction:
 - 'Kuala' : 0.1944
 - 'Syiah' : -0.1148
 - 'Laporan' : -0.0563
 - 'Anggota' : 0.0451
 - 'Kapolsek' : 0.0404
 - 'MELINTAS' : 0.0386

Interactive LIME HTML visualization saved to: /home/zapp/Kampus/PM-NEW/reports/explainability/lime_sample_explanation.html


In [14]:
# 15. ERROR-BASED EXPLAINABILITY

if ERROR_ANALYSIS_PATH.exists() and lime_ready:
    print("--- EXPLAINING A KNOWN ERROR ---")
    df_errors = pd.read_csv(ERROR_ANALYSIS_PATH)
    
    if len(df_errors) > 0:
        # Pick the first high-confidence error if available
        error_sample = df_errors.iloc[0]
        err_text = error_sample.iloc[0] # The text column
        
        print(f"Text: '{err_text}'")
        print(f"Actual (Human) Label: {error_sample['actual_label']}")
        print(f"Model Predicted: {error_sample['predicted_label']}")
        
        # Explain
        err_exp = explainer.explain_instance(
            err_text, 
            lime_predict_pipeline, 
            num_features=6, 
            top_labels=1
        )
        
        print("\nWhy did the model get it wrong? It focused heavily on these words:")
        pred_idx = err_exp.available_labels()[0]
        for feature, weight in err_exp.as_list(label=pred_idx):
            print(f" - '{feature}' : {weight:.4f}")
            
        err_html_path = EXPLAIN_DIR / "lime_error_explanation.html"
        err_exp.save_to_file(err_html_path)
        print(f"\nError explanation saved to: {err_html_path}")
    else:
        print("No errors found to explain!")
else:
    print("Skipped: Error analysis artifact or LIME not found.")


--- EXPLAINING A KNOWN ERROR ---
Text: 'tertawa lalu tb2 mnangis dulang2 sprti orgil'
Actual (Human) Label: harassment
Model Predicted: normal

Why did the model get it wrong? It focused heavily on these words:
 - 'orgil' : 0.1144
 - 'sprti' : -0.1104
 - 'tertawa' : 0.0880
 - 'lalu' : -0.0512
 - 'mnangis' : -0.0007
 - 'tb2' : -0.0005

Error explanation saved to: /home/zapp/Kampus/PM-NEW/reports/explainability/lime_error_explanation.html


In [15]:
# 16. GENERATE VISUALIZATIONS

# We will visualize the Top 10 words for the first 4 classes to keep the notebook clean
if is_linear:
    unique_classes = df_importance['Class'].unique()[:4]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx, class_name in enumerate(unique_classes):
        subset = df_importance[df_importance['Class'] == class_name].head(10)
        
        sns.barplot(
            data=subset, 
            x='Weight', 
            y='Feature', 
            ax=axes[idx], 
            palette='viridis'
        )
        axes[idx].set_title(f"Top Words for: {class_name}", fontweight='bold')
        axes[idx].set_xlabel("Importance Weight")
        axes[idx].set_ylabel("")
        
    plt.tight_layout()
    viz_path = EXPLAIN_DIR / "top_words_per_class.png"
    plt.savefig(viz_path, dpi=300)
    plt.close()
    print(f"Saved Global Word Importance visualization to {viz_path}")
    
elif is_tree:
    plt.figure(figsize=(10, 8))
    subset = df_importance.head(15)
    sns.barplot(data=subset, x='Weight', y='Feature', palette='mako')
    plt.title("Top Global Features (Tree Importances)", fontweight='bold')
    plt.xlabel("Gini / Split Weight")
    plt.ylabel("Word / N-gram")
    plt.tight_layout()
    
    viz_path = EXPLAIN_DIR / "top_global_words.png"
    plt.savefig(viz_path, dpi=300)
    plt.close()
    print(f"Saved Global Word Importance visualization to {viz_path}")


Saved Global Word Importance visualization to /home/zapp/Kampus/PM-NEW/reports/explainability/top_words_per_class.png


In [16]:
# 17. SAVE EXPLAINABILITY ARTIFACTS

# DataFrames and HTMLs are already saved progressively in the cells above.
print(f"All explainability artifacts verified and saved in: {EXPLAIN_DIR}")


All explainability artifacts verified and saved in: /home/zapp/Kampus/PM-NEW/reports/explainability


# 18. INTERPRETATION FRAMEWORK

When reading the CSVs and Visualizations from this notebook, apply the following logic:

1. **Association, Not Causation**: If the word "kamu" has a high weight for the class `Physical Bullying`, it does **not** mean the word "kamu" is a physical threat. It means that within this specific dataset, physical threats frequently contained the word "kamu". 
2. **Class Isolation (Linear Models)**: For Logistic Regression and LinearSVC, the `Class` column shows the target. High positive weight means the word strongly triggers a prediction *towards* that class.
3. **Lexical Shortcuts**: If you notice the model relying heavily on typos (e.g., "bgsd" instead of "bangsat"), the model is using lexical shortcuts. It memorize specific spelling variants rather than understanding semantics.

# 19. EXPLAINABILITY LIMITATIONS

As researchers, we must acknowledge the limitations of our methodology:
- **TF-IDF Blindness**: TF-IDF destroys word order. The phrase "tidak bodoh" might be split into "tidak" and "bodoh". If the unigram "bodoh" has a high weight for bullying, TF-IDF might misclassify the sentence despite the negation (unless captured by our Trigrams).
- **LIME Perturbation**: LIME works by generating thousands of slight variations of the input sentence and seeing how the prediction changes. It is an *approximation* of the model's logic locally, not an absolute proof of internal neural pathways.


# 20. EXPLAINABILITY SUMMARY

### Global Explanation
We successfully mapped the mathematical vectors of the selected model back to human-readable Indonesian vocabulary. We generated tables and charts displaying the most influential words driving the classification decisions for the cyberbullying categories.

### Local Explanation (LIME)
By implementing LIME, we created interactive HTML documents capable of highlighting the exact words within a sentence that pushed the AI toward its final conclusion. This is invaluable for qualitative error analysis and proves the model is not acting randomly.

### Next Step
The Data Science and Machine Learning pipeline is now **complete**. We have acquired data, preprocessed it, trained models, tuned them, evaluated them, analyzed their errors, and finally explained their logic. 

The next and final stage of the project is to build a User Interface to deploy this model: **`streamlit/app.py`**.


# 21. HOW TO RUN THIS NOTEBOOK

1. Activate your Python virtual environment.
2. If you haven't already, run `pip install lime matplotlib seaborn pandas scikit-learn` in your terminal.
3. Ensure previous notebooks (especially `06_tfidf.ipynb`, `09_model_evaluation.ipynb`, and `10_error_analysis.ipynb`) have been run successfully.
4. Open `notebooks/11_explainability.ipynb`.
5. Run the notebook from top to bottom.
6. In **Step 11**, review the DataFrames to see the top words for your model.
7. Navigate your file explorer to `reports/explainability/`.
8. **CRITICAL:** Double-click the `lime_sample_explanation.html` and `lime_error_explanation.html` files. They will open in your web browser and display a beautiful, color-coded highlight of the text showing exactly how the AI read the sentence.
9. Review the `.png` visualizations.
10. Congratulations on completing the ML pipeline! Proceed to develop the Streamlit app.
